# B3 — mixed displacement-stress: combining local refinement with the relative residual (attempt 6)

**A well-motivated combination, never tried before -- not a random grid search.** Attempt 4's own CPU root-cause investigation found exactly why the relative constitutive residual collapsed: its gradient w.r.t. `P_pred`, `2(P_pred-P_true)/(eps+P_true^2)`, shrinks roughly as `1/|P_true|` for large `|P_true|` -- precisely the groove-region points the residual was designed to stop from DOMINATING THE LOSS VALUE, at the cost of crippling their own GRADIENT. At the time, those groove points were represented by only the standard 6-8 quadrature points per element.

Attempt 5 (local integration refinement, just completed) gave those SAME groove-region points a real, measured improvement on the direct-P region QoI (824.9% -> 526.84% pooled RMS) by refining their quadrature to 1,000 points/element -- a much denser sample of exactly the high-stress region where attempt 4's gradient-vanishing problem was most damaging.

**This attempt tests whether the combination helps**: a denser quadrature at the groove might give the relative residual's otherwise-sound loss formula (it correctly stops high-stress points from dominating the loss VALUE -- the one thing it got right) enough aggregate gradient signal across many more points to avoid collapsing to a near-zero predictor, the specific failure mode attempt 4's own CPU diagnosis found and confirmed. Either outcome is a real, reportable finding: if it works, it directly validates the gradient-vanishing diagnosis as the right explanation and local refinement as a real fix for it; if it doesn't, the relative-residual formulation itself is a dead end regardless of integration resolution.

**No new code was needed** -- `mixed_stress_loss_B3_locally_refined` already accepts `relative_constitutive`/`constitutive_eps` (CPU-verified, `test_mixed_stress_B3_local_refine.py`) and `train_B3_mixed_stress.py`'s training loop already wires them through for the local-refine branch; this is a new combination of existing, already-tested flags, CPU-smoke-tested end to end before this notebook was built.

**Config**: attempt 4's own base (`p_output_scale=250`, `relative_constitutive=1`, `constitutive_eps=400.0`, `w_equilibrium=1.0`, no curriculum) plus `--local_refine_region 1 --local_refine_n_sub 10` (attempt 5's own setting). 50,000 iterations. **This notebook includes the same automated rate check (500 iterations) added after attempt 5's own real performance bug** -- it will stop with a clear error instead of running for hours if the rate looks wrong.

In [ ]:
# =====================================================================
#  CELL -- B3 mixed displacement-stress: attempt 6, combining attempt 4's
#  relative constitutive residual with attempt 5's local integration
#  refinement. See the markdown cell above for the full rationale.
# =====================================================================
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

import subprocess
import sys
import time

_started = time.time()


def run(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)


from google.colab import drive
if os.path.isdir('/content/drive/MyDrive'):
    print('Drive already mounted at /content/drive.')
else:
    drive.mount('/content/drive')

REPO = '/content/OMAR'
if not os.path.isdir(REPO):
    run(['git', 'clone', '-b', 'claude/claude-code-question-d307wp',
         'https://github.com/SUHIBAMRO/OMAR.git', REPO])
else:
    run(['git', '-C', REPO, 'fetch', 'origin', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'checkout', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'reset', '--hard', 'origin/claude/claude-code-question-d307wp'])

run([sys.executable, '-m', 'pip', 'install', '-q', 'pyvista<0.49'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-fem'])

WORK = f'{REPO}/Practical_Examples'
os.chdir(WORK)
sys.path.insert(0, WORK)

for _mod_name in list(sys.modules):
    if (_mod_name == 'omar_pfem' or _mod_name.startswith('omar_pfem.')
            or _mod_name == 'torchfem' or _mod_name.startswith('torchfem.')
            or _mod_name == 'pyvista' or _mod_name.startswith('pyvista.')):
        del sys.modules[_mod_name]

import torch
assert torch.cuda.is_available(), 'this cell needs a real GPU'
print('GPU:', torch.cuda.get_device_name(0))

import gc
for _attr in ('last_traceback', 'last_value', 'last_type'):
    if hasattr(sys, _attr):
        delattr(sys, _attr)
gc.collect()
torch.cuda.empty_cache()
print(f'GPU memory at start: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, '
      f'{torch.cuda.memory_reserved()/1e9:.2f} GB reserved (should be ~0 -- if not, '
      'use Runtime > Restart session before trusting this run)')

print(f'\n{"=" * 90}')
print('=== Re-running the CPU local-refine correctness checks in this environment ===')
from omar_pfem.test_mixed_stress_B3_local_refine import (
    test_empty_region_matches_plain_loss_exactly,
    test_nonempty_region_at_matching_degree_matches_plain_loss,
    test_finer_region_quadrature_changes_the_loss_and_stays_differentiable,
    test_production_n_sub_is_vectorized_not_a_python_loop,
)
test_empty_region_matches_plain_loss_exactly()
test_nonempty_region_at_matching_degree_matches_plain_loss()
test_finer_region_quadrature_changes_the_loss_and_stays_differentiable()
test_production_n_sub_is_vectorized_not_a_python_loop()

from omar_pfem.train_B3_mixed_stress import get_args, train
from omar_pfem.data.data_generate_B3_dataset import DEFAULT_RESOLUTION

device = torch.device('cuda')

R = '/content/drive/MyDrive/pfem_run'
OUTPUT_DIR = f'{R}/b3_mixed_stress_local_refine_relative'
os.makedirs(OUTPUT_DIR, exist_ok=True)

BASE_FLAGS = [
    '--pretrain_iters', '0',
    '--p_output_scale', '250.0',
    '--relative_constitutive', '1',
    '--constitutive_eps', '400.0',
    '--w_equilibrium', '1.0',
    '--local_refine_region', '1',
    '--local_refine_n_sub', '10',
    '--batch_size', '8',
]

# =====================================================================
#  Automated rate check (same safety net added for attempt 5, after a
#  real bug let a bad run go undetected for 21.7h of GPU time): a SHORT
#  run first, in a throwaway output dir, before the full 50,000.
# =====================================================================
EXPECTED_S_PER_ITER = 0.20
RATE_CHECK_ITERS = 500
RATE_CHECK_DIR = f'{OUTPUT_DIR}_ratecheck'
os.makedirs(RATE_CHECK_DIR, exist_ok=True)

print(f'\n{"=" * 90}')
print(f'=== Rate check: {RATE_CHECK_ITERS} iterations in a throwaway dir before the real run ===')
rate_check_args = get_args(BASE_FLAGS + [
    '--n_iters', str(RATE_CHECK_ITERS),
    '--log_every', str(RATE_CHECK_ITERS),
    '--ckpt_every', str(RATE_CHECK_ITERS),
    '--output_dir', RATE_CHECK_DIR,
])
t0 = time.time()
train(rate_check_args, device)
rate_check_elapsed = time.time() - t0
rate_check_s_per_iter = rate_check_elapsed / RATE_CHECK_ITERS
print(f'Rate check: {rate_check_elapsed:.1f}s for {RATE_CHECK_ITERS} iterations '
      f'= {rate_check_s_per_iter:.4f}s/iteration (expected ~{EXPECTED_S_PER_ITER}s/iteration).')
if rate_check_s_per_iter > 3 * EXPECTED_S_PER_ITER:
    raise RuntimeError(
        f'Rate check measured {rate_check_s_per_iter:.4f}s/iteration, more than 3x the expected '
        f'~{EXPECTED_S_PER_ITER}s/iteration -- STOPPING here instead of silently running the full '
        f'50,000 iterations for many hours -- report this back before re-running.')
print('Rate check OK -- proceeding to the real run.')

args = get_args(BASE_FLAGS + [
    '--n_iters', '50000',
    '--log_every', '500',
    '--ckpt_every', '5000',
    '--output_dir', OUTPUT_DIR,
])
n_elem = (DEFAULT_RESOLUTION[0] - 1) * (DEFAULT_RESOLUTION[1] - 1) * (DEFAULT_RESOLUTION[2] - 1)
print(f'\nTraining at {DEFAULT_RESOLUTION} -> {n_elem} elements, {args.n_iters} iterations, '
      f'p_output_scale={args.p_output_scale}, relative_constitutive={bool(args.relative_constitutive)}, '
      f'w_equilibrium={args.w_equilibrium}, local_refine_region={bool(args.local_refine_region)}, '
      f'local_refine_n_sub={args.local_refine_n_sub}, checkpoints -> {OUTPUT_DIR}')

torch.cuda.reset_peak_memory_stats(device)
t0 = time.time()
model = train(args, device)
elapsed_total = time.time() - t0

print(f'\n{"=" * 90}')
print(f'Training finished in {elapsed_total:.1f}s ({elapsed_total / args.n_iters:.4f}s/iteration average).')
print(f'GPU peak memory during training: '
      f'{torch.cuda.max_memory_allocated(device) / 1e6:.1f}MB allocated, '
      f'{torch.cuda.max_memory_reserved(device) / 1e6:.1f}MB reserved.')

# =====================================================================
#  Evaluation: same held-out FEM dataset every other B3 checkpoint uses.
# =====================================================================
print(f'\n{"=" * 90}')
print('=== Evaluating the final checkpoint against the held-out FEM dataset ===')
CHECKPOINT = f'{OUTPUT_DIR}/checkpoint_{args.n_iters}.pt'
DATASET = f'{R}/b3_dataset_clean_holdout/dataset.h5'
OUT_JSON = f'{OUTPUT_DIR}/qois_{args.n_iters}.json'

sys.argv = [
    'evaluate_B3_mixed_stress.py',
    '--checkpoint', CHECKPOINT,
    '--dataset', DATASET,
    '--out_json', OUT_JSON,
]
from omar_pfem.evaluate_B3_mixed_stress import main as evaluate_main
evaluate_main()

try:
    from omar_pfem.run_manifest import write_manifest
    write_manifest(OUTPUT_DIR, kind='b3_mixed_stress_local_refine_relative_training',
                    args={'n_iters': args.n_iters, 'p_output_scale': args.p_output_scale,
                          'relative_constitutive': bool(args.relative_constitutive),
                          'constitutive_eps': args.constitutive_eps,
                          'w_equilibrium': args.w_equilibrium,
                          'local_refine_region': bool(args.local_refine_region),
                          'local_refine_n_sub': args.local_refine_n_sub},
                    started_at=_started,
                    results={'elapsed_total_s': elapsed_total,
                             's_per_iter': elapsed_total / args.n_iters},
                    outputs=[CHECKPOINT, OUT_JSON],
                    notes="Item 1 attempt 6 (2026-10-26): combines attempt 4's relative "
                          "constitutive residual with attempt 5's local integration "
                          "refinement, testing whether denser groove-region quadrature "
                          "gives the relative residual enough gradient signal to avoid "
                          "the collapse-to-near-zero failure mode attempt 4's own CPU "
                          "diagnosis found. Never tried before; no new code, a new "
                          "combination of two already-separately-tested flags.")
except Exception as e:
    print(f'[manifest] not recorded: {e}')

print('\nDone. Send this cell\'s full printed output back -- especially '
      '"region_avg_sigma_xx_direct" pooled RMS -- compare against attempt 5\'s own 526.84% '
      'and attempt 4\'s own (corrected) 96.1% (itself a collapsed-predictor artifact, not '
      'genuine accuracy -- check the predicted-P mean/std too, not just the pooled RMS).')
